# Majority-class baseline

This notebook evaluates a simple baseline for arrival-delay classification. A separate majority-class classifier is fitted for each train type (IC, ICE, and RE) using January-June 2026 and evaluated on July-September 2026.

The baseline always predicts the most common target class observed in the training period for that train type. Macro-F1 is the primary metric because it gives each target class equal weight.

In [9]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from macro_f1_test import macro_f1_test

## Load the labelled data

Only the columns needed to construct the target and split the data are loaded. The source month is taken from the file name so that the split is chronological.

In [10]:
DATA_DIR = PROJECT_ROOT / "data"
files = sorted(DATA_DIR.glob("data-2026-0[1-9].parquet"))

if len(files) != 9:
    raise FileNotFoundError(
        f"Expected 9 monthly files in {DATA_DIR}, found {len(files)}. "
        "Run notebooks/setup.ipynb first."
    )

BASELINE_COLUMNS = [
    "delay_in_min",
    "arrival_is_canceled",
    "arrival_planned_time",
    "train_type",
]

parts = []
for file in files:
    frame = pd.read_parquet(file, columns=BASELINE_COLUMNS)
    frame = frame[frame["train_type"].isin(["IC", "ICE", "RE"])].copy()
    frame["source_file"] = file.name
    parts.append(frame)

data = pd.concat(parts, ignore_index=True)
data = data.dropna(subset=["arrival_planned_time"]).copy()
data["arrival_is_canceled"] = data["arrival_is_canceled"].fillna(False).astype(bool)

labelled = data[data["arrival_is_canceled"] | data["delay_in_min"].notna()].copy()
labelled["target"] = np.select(
    [
        labelled["arrival_is_canceled"],
        labelled["delay_in_min"] <= 1,
        labelled["delay_in_min"] <= 3,
        labelled["delay_in_min"] <= 10,
        labelled["delay_in_min"] <= 30,
    ],
    ["severe_or_cancelled", "on_time", "slight_delay", "late", "delayed"],
    default="severe_or_cancelled",
)

print(f"Labelled rows: {len(labelled):,}")
labelled["source_file"].value_counts().sort_index().to_frame("rows")

Labelled rows: 13,292,608


,rows
source_file,
data-2026-01.parquet,1591674
data-2026-02.parquet,1399550
data-2026-03.parquet,1483044
data-2026-04.parquet,1430929
data-2026-05.parquet,1472474
data-2026-06.parquet,1477827
data-2026-07.parquet,1446007
data-2026-08.parquet,1485514
data-2026-09.parquet,1505589


## Chronological train/test split

The baseline is fitted on January-June and evaluated on July-September. The test rows are never used when fitting the classifier.

In [11]:
train_files = {f"data-2026-{month:02d}.parquet" for month in range(1, 7)}
test_files = {f"data-2026-{month:02d}.parquet" for month in range(7, 10)}

train = labelled[labelled["source_file"].isin(train_files)].copy()
test = labelled[labelled["source_file"].isin(test_files)].copy()

print(f"Training rows: {len(train):,}")
print(f"Test rows: {len(test):,}")
print("\nTraining target distribution by train type:")
display(pd.crosstab(train["train_type"], train["target"], normalize="index").round(4))

Training rows: 8,855,498
Test rows: 4,437,110

Training target distribution by train type:


target,delayed,late,on_time,severe_or_cancelled,slight_delay
train_type,,,,,
IC,0.1758,0.2301,0.2731,0.1891,0.1319
ICE,0.1917,0.2038,0.2948,0.1887,0.1210
RE,0.0981,0.2081,0.4649,0.0556,0.1733


## Majority-class baseline by train type

Each train type receives its own fitted `DummyClassifier`. A constant feature is used because the baseline does not learn from input features; it only learns the most frequent training label.

In [12]:
results = []
fitted_models = {}

for train_type in ["IC", "ICE", "RE"]:
    train_group = train[train["train_type"] == train_type]
    test_group = test[test["train_type"] == train_type]

    if train_group.empty or test_group.empty:
        raise ValueError(f"Missing training or test rows for train type: {train_type}")

    model = DummyClassifier(strategy="most_frequent")
    model.fit(np.zeros((len(train_group), 1)), train_group["target"])
    fitted_models[train_type] = model

    result = macro_f1_test(
        model,
        np.zeros((len(test_group), 1)),
        test_group["target"],
        model_name=f"majority_{train_type}",
    )
    result["train_type"] = train_type
    result["training_majority_class"] = train_group["target"].value_counts().idxmax()
    results.append(result)

baseline_results = (
    pd.DataFrame(results)
    [["train_type", "model", "training_majority_class", "n_test", "macro_f1", "accuracy"]]
    .sort_values("train_type")
    .reset_index(drop=True)
)
baseline_results

,train_type,model,training_majority_class,n_test,macro_f1,accuracy
0,IC,majority_IC,on_time,154540,0.084367,0.267296
1,ICE,majority_ICE,on_time,520527,0.077333,0.239669
2,RE,majority_RE,on_time,3762043,0.123175,0.444955


The table above is the reference macro-F1 and accuracy for a majority-class prediction within each train type. Later models should be evaluated on the same test groups and compared against these values.